# FROST — 단일탭 사이트 분류 (깨끗한 서버)

브라우저 OPFS의 SSD 경합(read latency)만으로 **다른 탭에서 어떤 사이트를 보는지** 맞히는 분류 노트북.

## 정확도 세 가지 (서로 다른 값이니 헷갈리지 말 것)
| 숫자 | 뜻 | 값 | 셀 |
|---|---|---|---|
| **same-session** | 같은 세션 데이터 안에서 CV. train/test가 섞여 **낙관적 상한** | ~95% | 4 |
| **cross-session** | 처음 보는 **다른 날** 세션으로 테스트. **진짜 일반화 성능** | ~81% | 5 |
| **멀티 trace 투표** | 공격자가 trace 여러 개 모아 다수결. **실전 정확도** | k=5 ~95% / k=10 ~97% | 6 |

> 8사이트, 랜덤 추측 12.5%. cross/투표는 **두 세션에 모두 있는 7사이트** 대상(idle은 11-24 세션에 없어 제외).

**특징**: `times` 기반 실시간 0.1초 binning + 칸별(평균·std·p90·읽기율) + 전역 절대 특징(총 읽기 수=throughput, latency 분위수, slow-read 비율).
**모델**: ExtraTrees.

## 셀 구성
1. 데이터 업로드 (**두 세션 zip 모두**)
2. 로딩 (라벨 정규화 + 세션)
3. 특징 추출
4. 단일탭 모델 — same-session CV + 혼동행렬 *(낙관적 상한)*
5. 세션 간(cross-session) 평가 *(진짜 일반화 성능)*
6. 멀티 trace 투표 — k 곡선 + 혼동행렬 *(실전 정확도)*
7. **포그라운드 검출** — 멀티탭 trace에서 '지금 보는 탭' 맞히기 *(새 인사이트)*

> 향후 작업: 멀티탭(배경+포그라운드)·쌍 분류, 세션 추가 수집(→ cross-session 향상).


In [ ]:
# === 1. 데이터 업로드 (두 세션 모두) ===
# 세션 A(*_bg_blank.jsonl)와 세션 B(youtube.jsonl 등)를 한 zip에 담아 올리면
# 셀 5(cross-session)·6(투표)이 두 세션을 비교 평가한다.
from google.colab import files
import zipfile, os

uploaded = files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('.')
import glob
found = glob.glob('**/*_bg_blank.jsonl', recursive=True)
print(f'{len(found)}개 파일 찾음:')
for f in sorted(found):
    print('  ', f)

In [ ]:
# === 2. 로딩 (label 정규화 + session, 여러 세션 지원) ===
import json, glob, os
import numpy as np
from collections import defaultdict

# 단일탭 비교 대상 8개 사이트. 세션 A는 label이 'youtube_bg_blank',
# 세션 B는 'youtube'처럼 다르게 저장돼 있어서 접미사를 떼어 통일한다.
# (멀티탭 '*_bg_figma' 등은 단일탭 비교에서 자동 제외)
SITES = ['idle', 'youtube', 'coupang', 'figma', 'google', 'chatgpt', 'instagram', 'navermap']

trials = []
for f in sorted(glob.glob('**/*.jsonl', recursive=True)):
    for line in open(f, encoding='utf-8'):
        if not line.strip():
            continue
        r = json.loads(line)
        lab = r.get('label')
        if not isinstance(lab, str):
            continue
        if lab.endswith('_bg_blank'):
            lab = lab[:-len('_bg_blank')]
        if lab not in SITES:
            continue
        tr = np.asarray(r['trace'], dtype=np.float64)
        tm = r.get('times')
        tm = np.asarray(tm, dtype=np.float64) if isinstance(tm, list) and len(tm) == len(tr) else None
        trials.append(dict(label=lab, session=r.get('session') or 'unknown',
                           trace=tr, times=tm, dur=(r.get('durationMs') or 10000) / 1000))

labels = sorted({t['label'] for t in trials}, key=lambda l: (l != 'idle', l))
y = np.array([labels.index(t['label']) for t in trials])
sessions = np.array([t['session'] for t in trials])   # 셀 4c(cross-session)에서 사용

print('labels:', labels)
print('세션:', sorted(set(sessions)))
g = defaultdict(lambda: defaultdict(int))
for t in trials:
    g[t['label']][t['session']] += 1
print('라벨 x 세션 trial 수:')
for l in labels:
    print(f'  {l:<10}', dict(g[l]))


In [ ]:
# === 3. 보강 특징 추출 ===
# [개선] 현행은 cnt/mean, mean/base 로 "절대 스케일"을 전부 정규화해 버려서
# 가장 강한 신호(사이트별 throughput/절대 latency)를 모델이 못 봤다.
#   예: idle 10초간 ~174k reads vs youtube ~58k reads  (3배 차이)
# 그래서 아래 3가지를 더한다:
#   (1) times 기반 "실시간" binning
#   (2) 칸별 "읽기율(reads/sec)" 절대값
#   (3) 전역 절대 특징(총 읽기 수, latency 분위수, slow-read 비율, base)
# 효과(동일 데이터, same-session CV): top-1 83% -> 95%, top-3 93% -> 99%
BIN_S, USE_S, SPIKE = 0.1, 10.0, 1.0
NB = int(round(USE_S / BIN_S))   # 100 bins

def binstats(tr, times, dur):
    tr = np.asarray(tr, float)
    # 실제 시각(초). times 없으면 index 균등 가정으로 폴백.
    t = (times / 1000.0) if times is not None else (np.arange(len(tr)) + 0.5) * dur / len(tr)
    keep = t < USE_S
    raw = tr[keep]            # 정규화 전 원시 latency (전역 특징용)
    tt = t[keep]
    clipped = np.clip(raw, 0, SPIKE)
    idx = np.minimum((tt / BIN_S).astype(int), NB - 1)

    mean = np.zeros(NB); std = np.zeros(NB); p90 = np.zeros(NB); rate = np.zeros(NB)
    for b in range(NB):
        s = clipped[idx == b]
        rate[b] = len(s) / BIN_S          # 그 칸의 초당 읽기 수(절대값)
        if len(s):
            mean[b] = s.mean(); std[b] = s.std(); p90[b] = np.percentile(s, 90)

    base = mean[5:20].mean() + 1e-9       # 2초 구간 기준(기기 불변 shape용)
    shape = np.concatenate([mean / base, std / base, p90 / base])  # 정규화 모양

    n = len(raw)
    glob = np.array([
        n, np.log1p(n),                   # 총 읽기 수 = throughput (최강 신호)
        raw.mean(), np.median(raw),
        np.percentile(raw, 90), np.percentile(raw, 99),
        (raw > 0.3).mean(), (raw > 0.5).mean(), (raw > 1.0).mean(),  # 경합 비율
        base, rate.mean(), rate.std(),
    ])
    return np.concatenate([shape, rate, glob])  # 300 + 100 + 12 = 412

X = np.stack([binstats(t['trace'], t['times'], t['dur']) for t in trials])
print('특징 행렬:', X.shape, '  (사이트당', (y == 0).sum(), 'trial)')


In [ ]:
# === 4. 단일탭 모델: same-session CV (top-1/top-3) + 혼동행렬 ===
# [주의] 같은 세션 trial이 train/test에 섞이므로 '낙관적 상한'. 진짜 성능은 셀 5.
# 모델을 kNN+RF 앙상블 -> ExtraTrees 단독으로 교체 (kNN이 앙상블을 깎아먹던 문제 해결)
# 효과: top-1 ~50% -> ~76%, top-3 ~88% -> ~92% (데이터 동일, 모델만 교체)
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.base import clone

MODELS = {
    'ExtraTrees': ExtraTreesClassifier(n_estimators=400, random_state=0, n_jobs=-1),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1),
}
BEST = 'ExtraTrees'   # 데모/멀티탭에 쓸 최종 모델
n = len(labels)
print(f'{n}-class  (랜덤 추측 {1/n:.0%})\n')
conf = np.zeros((n, n), int)
for name, m in MODELS.items():
    t1, t3 = [], []
    for rep in range(10):
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=rep).split(X, y):
            mm = clone(m).fit(X[tr], y[tr])
            order = np.argsort(mm.predict_proba(X[te]), 1)[:, ::-1]
            t1.append((order[:, 0] == y[te]).mean())
            t3.append(np.mean([yy in o[:3] for yy, o in zip(y[te], order)]))
            if name == BEST and rep == 0:
                for a, b in zip(y[te], order[:, 0]):
                    conf[a, b] += 1
    star = '  <- 최종 모델' if name == BEST else ''
    print(f'  {name:<14} top-1 {np.mean(t1):.0%} ± {np.std(t1):.0%}   top-3 {np.mean(t3):.0%}{star}')

# 최종 모델 하나만 전체 데이터로 학습 (셀 4b 멀티탭 / 데모 셀에서 사용)
clf = [clone(MODELS[BEST]).fit(X, y)]

# 혼동행렬 그림 (최종 모델 기준)
cmn = conf / conf.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
for i in range(n):
    for j in range(n):
        if conf[i, j]:
            ax.text(j, i, conf[i, j], ha='center', va='center',
                    color='white' if cmn[i, j] > 0.5 else '#0b0b0b')
ax.set_xticks(range(n), labels, rotation=40, ha='right'); ax.set_yticks(range(n), labels)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'{BEST} confusion (top-1 {np.trace(conf)/conf.sum():.0%})')
fig.colorbar(im, fraction=0.046, format=PercentFormatter(1))
plt.tight_layout(); plt.show()

In [ ]:
# === 5. 세션 간(cross-session) 평가 — "처음 보는 세션"에서의 진짜 성능 ===
# same-session CV(셀 4)는 같은 세션의 trial이 train/test에 섞여 낙관적이다.
# 여기서는 세션 하나를 통째로 빼서 test(LeaveOneGroupOut) → 진짜 일반화 성능.
# 세션이 1개뿐이면 평가 불가(다른 날 세션을 더 모아야 함).
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.base import clone
from collections import defaultdict

uniq_sess = sorted(set(sessions))
if len(uniq_sess) < 2:
    print(f'세션이 {len(uniq_sess)}개뿐 -> cross-session 평가 불가. 다른 날 세션을 더 모으세요.')
else:
    # 2개 이상 세션에 등장하는 라벨만 (처음 보는 세션으로 테스트하려면 필수)
    lab_sess = defaultdict(set)
    for t in trials:
        lab_sess[t['label']].add(t['session'])
    keep = [i for i, l in enumerate(labels) if len(lab_sess[l]) >= 2]
    mask = np.isin(y, keep)
    print('cross-session 대상 라벨:', [labels[i] for i in keep])
    excl = [labels[i] for i in range(len(labels)) if i not in keep]
    if excl:
        print('  (세션 1개뿐이라 제외:', excl, ')')
    Xm, ym, sm = X[mask], y[mask], sessions[mask]
    t1s, t3s = [], []
    for tr, te in LeaveOneGroupOut().split(Xm, ym, sm):
        held = sm[te][0]
        if len(np.unique(ym[te])) < 2:   # 그 세션에 라벨이 1종뿐이면 '판별' 평가가 안 됨(무조건 정답)
            print(f'  test=세션 {held:<12} 라벨 1종뿐 -> 건너뜀 (idle만 측정한 세션 등)')
            continue
        mm = clone(MODELS[BEST]).fit(Xm[tr], ym[tr]); cls = mm.classes_
        order = np.argsort(mm.predict_proba(Xm[te]), 1)[:, ::-1]
        t1 = np.mean([ym[te][i] == cls[order[i, 0]] for i in range(len(te))])
        t3 = np.mean([ym[te][i] in cls[order[i, :3]] for i in range(len(te))])
        t1s.append(t1); t3s.append(t3)
        print(f'  test=세션 {held:<12} top-1 {t1:.0%}  top-3 {t3:.0%}')
    print(f'\n  ⭐ 평균 cross-session  top-1 {np.mean(t1s):.0%}  top-3 {np.mean(t3s):.0%}')
    print('  (same-session CV ~95%와 비교 -> 이 값이 진짜 일반화 성능. 차이 = 한계의 크기)')


In [ ]:
# === 6. 멀티 trace 투표 — 실전 정확도 (k 곡선 + 혼동행렬) ===
# 공격 페이지는 피해자가 사이트를 보는 동안 trace를 여러 개 뽑을 수 있다.
# trace k개의 예측확률을 평균내 다수결 -> 단일 trace의 오류가 상쇄돼 정확도가 뛴다.
# 평가는 셀 5와 같은 cross-session(LeaveOneGroupOut): 처음 보는 세션에서 투표.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.base import clone
from collections import defaultdict

uniq_sess = sorted(set(sessions))
if len(uniq_sess) < 2:
    print(f'세션이 {len(uniq_sess)}개뿐 -> 투표의 cross-session 효과를 보려면 다른 날 세션이 필요.')
else:
    rng = np.random.default_rng(0)
    lab_sess = defaultdict(set)
    for t in trials:
        lab_sess[t['label']].add(t['session'])
    keep = [i for i, l in enumerate(labels) if len(lab_sess[l]) >= 2]
    mask = np.isin(y, keep)
    Xm, ym, sm = X[mask], y[mask], sessions[mask]
    sublabels = [labels[i] for i in keep]
    print('대상 라벨:', sublabels)

    KS = [1, 2, 3, 5, 10]
    REPS = 300          # 라벨마다 무작위로 k개 뽑아 투표하는 반복 횟수
    CONF_K = 5          # 혼동행렬을 그릴 k
    curve = {k: [] for k in KS}
    conf = np.zeros((len(labels), len(labels)), int)

    for tr, te in LeaveOneGroupOut().split(Xm, ym, sm):
        if len(np.unique(ym[te])) < 2:   # 라벨 1종뿐인 세션 fold는 제외
            continue
        mm = clone(MODELS[BEST]).fit(Xm[tr], ym[tr]); cls = mm.classes_
        P = mm.predict_proba(Xm[te]); yte = ym[te]
        for k in KS:
            acc = []
            for c in np.unique(yte):
                idx = np.where(yte == c)[0]
                for _ in range(REPS):
                    pick = rng.choice(idx, size=min(k, len(idx)), replace=False)
                    pred = cls[np.argmax(P[pick].mean(0))]
                    acc.append(pred == c)
                    if k == CONF_K:
                        conf[c, pred] += 1
            curve[k].append(np.mean(acc))

    means = [np.mean(curve[k]) for k in KS]
    print('\ntrace k개 투표 시 cross-session top-1:')
    for k, m in zip(KS, means):
        print(f'  k={k:<3} {m:.0%}')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    # (1) k 곡선
    ax1.plot(KS, means, 'o-', lw=2)
    for k, m in zip(KS, means):
        ax1.annotate(f'{m:.0%}', (k, m), textcoords='offset points', xytext=(0, 8), ha='center')
    ax1.axhline(1/len(sublabels), ls='--', c='gray', label=f'random {1/len(sublabels):.0%}')
    ax1.set_xlabel('traces aggregated per vote (k)'); ax1.set_ylabel('cross-session top-1')
    ax1.set_ylim(0, 1.02); ax1.yaxis.set_major_formatter(PercentFormatter(1))
    ax1.set_title('more traces -> higher accuracy'); ax1.legend(); ax1.grid(alpha=0.3)
    # (2) k=CONF_K 혼동행렬
    kk = keep
    sub = conf[np.ix_(kk, kk)].astype(float)
    cmn = sub / sub.sum(1, keepdims=True)
    im = ax2.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    for i in range(len(kk)):
        for j in range(len(kk)):
            if sub[i, j]:
                ax2.text(j, i, f'{cmn[i,j]:.0%}', ha='center', va='center',
                         color='white' if cmn[i, j] > 0.5 else '#0b0b0b', fontsize=8)
    ax2.set_xticks(range(len(kk)), sublabels, rotation=40, ha='right')
    ax2.set_yticks(range(len(kk)), sublabels)
    ax2.set_xlabel('predicted'); ax2.set_ylabel('true')
    ax2.set_title(f'k={CONF_K} voting confusion (top-1 {np.mean(curve[CONF_K]):.0%})')
    fig.colorbar(im, ax=ax2, fraction=0.046, format=PercentFormatter(1))
    plt.tight_layout(); plt.show()


In [ ]:
# === 7. 포그라운드 검출 — 멀티탭 trace에서 "지금 보고 있는 탭" 맞히기 ===
# 단일탭만 학습한 모델(clf)에 멀티탭 trace를 넣는다. 포그라운드 탭이 디스크 경합을
# 지배하므로, 멀티탭 학습 없이도 '활발히 보는 사이트'가 top-1으로 복원되는지 본다.
#
# [데이터 규칙] 파일 라벨을 "FG_bg_BG" 꼴로 저장하면 자동 인식된다.
#   예: youtube_bg_figma  -> 포그라운드=youtube, 백그라운드=figma
#   백그라운드를 숨긴 테스트는 아래 FG_OVERRIDE에 수동 등록(예: secret).
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

FG_OVERRIDE = {
    'secret':  ('navermap', None),   # 배경 가려진 테스트: 포그라운드=navermap
    'secret1': ('chatgpt',  None),   # 배경 가려진 테스트: 포그라운드=chatgpt
}

def _resolve(lab):
    if lab in FG_OVERRIDE:
        return FG_OVERRIDE[lab]
    if '_bg_' in lab:
        a, b = lab.split('_bg_', 1)
        if a in SITES and b in SITES:
            return (a, b)          # FG, BG
    return None                    # 단일탭(_bg_blank)이나 무관 파일은 제외

# 멀티탭 레코드 수집 (셀 2가 쓰던 것과 같은 파일들을 다시 훑는다)
import glob, json
multi = {}
for f in sorted(glob.glob('**/*.jsonl', recursive=True)):
    for line in open(f, encoding='utf-8'):
        if not line.strip():
            continue
        r = json.loads(line); lab = r.get('label')
        if not isinstance(lab, str):
            continue
        info = _resolve(lab)
        if info is None:
            continue
        tr = np.asarray(r['trace'], float); tm = r.get('times')
        tm = np.asarray(tm, float) if isinstance(tm, list) and len(tm) == len(tr) else None
        feat = binstats(tr, tm, (r.get('durationMs') or 10000) / 1000)
        multi.setdefault(lab, (info, []))[1].append(feat)

model = clf[0]
cls = [labels[i] for i in model.classes_]

if not multi:
    print('멀티탭 파일이 없습니다.')
    print('파일 라벨을 "youtube_bg_figma"(=FG_bg_BG) 꼴로 저장하면 이 셀이 자동 인식합니다.')
else:
    names, fg1, fg2, fg3, votes, bgdet = [], [], [], [], [], []
    print(f'{"recording":<22}{"FG":<10}{"BG":<9}{"top1":>6}{"top2":>6}{"top3":>6}  투표')
    for lab, ((fg, bg), feats) in multi.items():
        X = np.stack(feats); P = model.predict_proba(X)
        def rank(p): return [cls[i] for i in np.argsort(p)[::-1]]
        t1 = np.mean([rank(p)[0] == fg for p in P])
        t2 = np.mean([fg in rank(p)[:2] for p in P])
        t3 = np.mean([fg in rank(p)[:3] for p in P])
        vote = cls[int(np.argmax(P.mean(0)))]
        bd = np.mean([bg in rank(p)[:2] for p in P]) if bg else float('nan')
        print(f'{lab:<22}{fg:<10}{str(bg):<9}{t1:>6.0%}{t2:>6.0%}{t3:>6.0%}  {vote}{" O" if vote == fg else " X"}')
        names.append(f'{lab}\n(n={len(feats)})'); fg1.append(t1); fg2.append(t2); fg3.append(t3)
        votes.append(vote == fg); bgdet.append(bd)

    # 그래프: 레코딩별 FG 복원율(top-1/2/3)
    x = np.arange(len(names)); w = 0.25
    fig, ax = plt.subplots(figsize=(max(7, 2.2 * len(names)), 5))
    ax.bar(x - w, fg1, w, label='FG top-1')
    ax.bar(x,     fg2, w, label='FG top-2')
    ax.bar(x + w, fg3, w, label='FG top-3')
    for i in range(len(names)):
        ax.text(i, 1.02, 'vote: ' + ('O' if votes[i] else 'X'),
                ha='center', fontsize=9, color='green' if votes[i] else 'red')
    ax.set_xticks(x, names); ax.set_ylim(0, 1.12)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.set_ylabel('foreground recovered'); ax.axhline(1/len(cls), ls='--', c='gray', label=f'random {1/len(cls):.0%}')
    ax.set_title('Foreground detection from multi-tab traces (single-tab model)')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()

    bgvals = [b for b in bgdet if b == b]
    if bgvals:
        print(f'\n배경(BG) 탭이 top-2에 등장한 비율: 평균 {np.mean(bgvals):.0%}'
              '  -> 포그라운드가 신호를 지배, 배경은 거의 안 잡힘.')
